In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data Collection

In [ ]:
data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/344-331/mini_project/Depression Student Dataset.csv')

In [ ]:
data.sample(10)

In [ ]:
data.shape

In [ ]:
data.columns

# Data cleaning

## Data Inspection

In [ ]:
#ตรวจสอบชนิดข้อมูลแต่ละคอลัมน์
data.dtypes

In [ ]:
#ตรวจสอบข้อมูลทางสถิติ
data.describe()

In [ ]:
#ตรวจสอบว่าแต่ละคอลัมน์มี Missing value เท่าไหร่
data.isnull().sum()

## Inconsistent Data

In [ ]:
#ตรวจสอบว่าคอลัมน์ Gender มีค่าอะไรบ้าง
data['Gender'].unique()

In [ ]:
#ตรวจสอบว่าคอลัมน์ Have you ever had suicidal thoughts ? มีค่าอะไรบ้าง
data['Have you ever had suicidal thoughts ?'].unique()

In [ ]:
#ตรวจสอบว่าคอลัมน์ Family History of Mental Illness มีค่าอะไรบ้าง
data['Family History of Mental Illness'].unique()

In [ ]:
#ตรวจสอบว่าคอลัมน์ Depression มีค่าอะไรบ้าง
data['Depression'].unique()

## Data Integrity

In [ ]:
#ตรวสอบว่ามีข้อมูลในแถวของคอลัมน์ Age ใดบ้างที่เป็นค่าติดลบ
data[data['Age'] < 0]

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=data.select_dtypes(include=[np.number]))
plt.xticks(rotation=45)
plt.title("Outliers")
plt.show()

## Feature Engineering

In [ ]:
data['Sleep Duration'].unique()

In [ ]:
# แปลง Sleep Duration เป็นตัวเลข
def convert_sleep_duration(duration):
    if duration == 'Less than 5 hours':
        return 3.5
    elif duration == '5-6 hours':
        return 5.5
    elif duration == '7-8 hours':
        return 7.5
    elif duration == 'More than 8 hours':
        return 9.5
data['Sleep Duration Numeric'] = data['Sleep Duration'].apply(convert_sleep_duration)

In [ ]:
# สร้างคอลัมน์ใหม่สำหรับการจัดกลุ่มการนอน
'''def categorize_sleep(hours):
    if pd.isna(hours):
        return 'Unknown'
    elif hours < 5:
        return 'Low Sleep'
    elif 5 <= hours <= 8:
        return 'Normal Sleep'
    else:
        return 'High Sleep'

data['Sleep Category'] = data['Sleep Duration Numeric'].apply(categorize_sleep)'''

In [ ]:
data.columns

In [ ]:
#แปลง Study Hours เป็นช่วง
def convert_sleep_hours(hours):
  if 10 <= hours <= 12:
    return '10-12'
  elif 7 <= hours <= 9:
    return '7-9'
  elif 4 <= hours <= 6:
    return '4-6'
  else:
    return '0-3'

data['Sleep Hours Category'] = data['Study Hours'].apply(convert_sleep_hours)

In [ ]:
data

# Exploratory Data Analysis - EDA

## Numerical Data Distribution

In [ ]:
num_cols = data.select_dtypes(include=['number']).columns
num_cols

In [ ]:
colors = sns.color_palette('pastel')
colors

In [ ]:
fig, axes = plt.subplots(nrows=len(num_cols), ncols=1, figsize=(12, 10))

for i, col in enumerate(num_cols):
    data[col].hist(ax=axes[i], bins=30, color=colors[i % len(colors)])
    axes[i].set_title(col)

plt.suptitle('Numerical Data Distribution')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = data.select_dtypes(include=['object']).columns
num_colors = len(colors)
for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=data[col], palette='viridis')
    plt.title(f"Categorical Variable Distribution {col}")
    plt.show()

## Correlation Matrix

In [ ]:
'''plt.figure(figsize=(10, 6))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()'''

## Correlation Between Each Feature and Depression

In [ ]:
# Gender & Depression
'''plt.figure(figsize=(8, 6))
sns.countplot(x='Gender', hue='Depression', data=data)
plt.title('Gender vs Depression')
plt.show()'''

In [ ]:
male_depression_counts = data[(data['Gender'] == 'Male') & (data['Depression'] == 'Yes')].shape[0]
male_no_depression_counts = data[(data['Gender'] == 'Male') & (data['Depression'] == 'No')].shape[0]

female_depression_counts = data[(data['Gender'] == 'Female') & (data['Depression'] == 'Yes')].shape[0]
female_no_depression_counts = data[(data['Gender'] == 'Female') & (data['Depression'] == 'No')].shape[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))  # 1 row, 2 columns

labels = ['Depression', 'No Depression']
sizes_male = [male_depression_counts, male_no_depression_counts]
colors = ['lightcoral', 'lightskyblue']
explode = (0.1, 0)

ax1.pie(sizes_male, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
ax1.set_title('Proportion of Males with and without Depression')
ax1.axis('equal')

sizes_female = [female_depression_counts, female_no_depression_counts]
ax2.pie(sizes_female, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
ax2.set_title('Proportion of Females with and without Depression')
ax2.axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Age & Depression
plt.figure(figsize=(10, 6))
sns.boxplot(x='Depression', y='Age', data=data , palette=colors)
plt.title('Age vs Depression')
plt.show()

In [ ]:
depression_age_means = data.groupby('Depression')['Age'].mean()

plt.figure(figsize=(10, 6))
sns.boxplot(x='Depression', y='Age', data=data, palette=colors[::-1])
plt.title('Age vs Depression')

for depression_group, age_mean in depression_age_means.items():
    plt.text(
        x=depression_group,
        y=age_mean + 1,
        s=f'Avg: {age_mean:.2f}',
        ha='center',
        va='bottom',
        color='black',
        fontsize=10,
    )

plt.show()

In [ ]:
# Age & Depression
plt.figure(figsize=(5, 6))
depression_data = data[data['Depression'] == 'Yes']
sns.boxplot(y='Age', data=depression_data, palette=colors)
plt.title('Age Distribution for Individuals with Depression')
plt.ylabel('Age')
plt.show()

In [ ]:
# Academic Pressure & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Academic Pressure', hue='Depression', data=data)
plt.title('Academic Pressure vs Depression')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ... (Your existing code) ...

# Academic Pressure & Depression (Stacked Percentage Bar)
academic_pressure_depression = data.groupby(['Academic Pressure', 'Depression']).size().unstack(fill_value=0)

# Calculate percentages for stacked bar
academic_pressure_depression_perc = academic_pressure_depression.div(academic_pressure_depression.sum(axis=1), axis=0) * 100

# Create stacked bar plot
ax = academic_pressure_depression_perc.plot(kind='bar', stacked=True, figsize=(10, 6), color=['lightskyblue', 'lightcoral'])  # Change stacked to True

plt.title('Percentage of Depression by Academic Pressure (Stacked)')
plt.xlabel('Academic Pressure')
plt.ylabel('Percentage')
plt.xticks(rotation=0)
plt.legend(title='Depression')

# Display percentage labels inside stacked bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:  # Avoid displaying labels for 0% values
        ax.annotate(f'{height:.1f}%', (x + width / 2, y + height / 2), ha='center', va='center', color='white')  # Add color='white' for visibility

plt.tight_layout()
plt.show()

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
academic_pressure_depression = data.groupby(['Academic Pressure', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม (ในแนวตั้ง)
academic_pressure_percent = academic_pressure_depression.div(academic_pressure_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
academic_pressure_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Number of Individuals with Depression by Academic Pressure (Stacked)')
plt.xlabel('Academic Pressure')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์บนกราฟ
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = academic_pressure_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white')

plt.tight_layout()
plt.show()


In [ ]:
# Study Satisfaction & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Study Satisfaction', hue='Depression', data=data)
plt.title('Study Satisfaction vs Depression')
plt.show()

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
study_satisfaction_depression = data.groupby(['Study Satisfaction', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
study_satisfaction_percent = study_satisfaction_depression.div(study_satisfaction_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
study_satisfaction_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Study Satisfaction vs Depression (Stacked Bar Chart)')
plt.xlabel('Study Satisfaction')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = study_satisfaction_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
sleep_depression = data.groupby(['Sleep Category', 'Depression']).size().unstack(fill_value=0)

sleep_depression_prop = sleep_depression.div(sleep_depression.sum(axis=1), axis=0)

sleep_depression_prop.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Proportion of Normal Sleep for those with and without Depression')
plt.xlabel('Sleep Category')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(title='Depression')
plt.tight_layout()
plt.show()


In [ ]:
data

In [ ]:
# Sleep Duration Numeric & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Sleep Duration Numeric', hue='Depression', data=data)
plt.title('Sleep Category vs Depression')
plt.show()

In [ ]:
data['Sleep Duration'].unique()

# 66666

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
sleep_duration_depression = data.groupby(['Sleep Duration', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
sleep_duration_percent = sleep_duration_depression.div(sleep_duration_depression.sum(axis=1), axis=0) * 100

# เรียงลำดับข้อมูลตามที่ต้องการ
desired_order = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
study_hours_depression = study_hours_depression.reindex(desired_order, axis=0)

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
sleep_duration_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Sleep Duration vs Depression (Stacked Bar Chart)')
plt.xlabel('Sleep Duration (Hours)')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = sleep_duration_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Dietary Habits & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Dietary Habits', hue='Depression', data=data)
plt.title('Dietary Habits vs Depression')
plt.show()

In [ ]:
'''# Dietary Habits & Depression
dietary_habits = data['Dietary Habits'].unique()  # Get unique dietary habits

fig, axes = plt.subplots(1, len(dietary_habits), figsize=(18, 6))

for i, habit in enumerate(dietary_habits):
    # Filter data for the current dietary habit
    habit_data = data[data['Dietary Habits'] == habit]

    colors = ['lightskyblue', 'lightcoral']

    # Count depression cases for the current habit
    depression_counts = habit_data['Depression'].value_counts()

    # Create the pie chart for the current habit
    axes[i].pie(depression_counts, labels=depression_counts.index, autopct='%1.1f%%', startangle=90, colors=colors, shadow=True)
    axes[i].set_title(f'Depression Proportion for {habit}')
    axes[i].axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

plt.tight_layout()
plt.show()'''


In [ ]:
import matplotlib.pyplot as plt

# Unique dietary habits
dietary_habits = data['Dietary Habits'].unique()

fig, axes = plt.subplots(1, len(dietary_habits), figsize=(18, 6))

colors = ['lightskyblue', 'lightcoral']
labels = ['No Depression', 'Yes Depression']  # Custom labels

for i, habit in enumerate(dietary_habits):
    habit_data = data[data['Dietary Habits'] == habit]
    depression_counts = habit_data['Depression'].value_counts()

    # Create the pie chart
    wedges, _, _ = axes[i].pie(depression_counts, autopct='%1.1f%%', startangle=90, colors=colors, shadow=True)

    axes[i].set_title(f'Depression Proportion for {habit}')
    axes[i].axis('equal')

# Add legend **outside** the plot (ขวาของกราฟ)
fig.legend(wedges, labels, loc="center left", bbox_to_anchor=(1, 0.5), title="Depression Status")

plt.tight_layout()
plt.show()


In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
dietary_habits_depression = data.groupby(['Dietary Habits', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
dietary_habits_percent = dietary_habits_depression.div(dietary_habits_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
dietary_habits_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Number of Individuals with Depression by Dietary Habits')
plt.xlabel('Dietary Habits')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=15)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์บนกราฟ
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = dietary_habits_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white')

plt.tight_layout()
plt.show()


In [ ]:
# Have you ever had suicidal thoughts ? & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Have you ever had suicidal thoughts ?', hue='Depression', data=data)
plt.title('Have you ever had suicidal thoughts ? vs Depression')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
male_suicidal_counts = data[(data['Gender'] == 'Male') & (data['Have you ever had suicidal thoughts ?'] == 'Yes')].shape[0]
male_no_suicidal_counts = data[(data['Gender'] == 'Male') & (data['Have you ever had suicidal thoughts ?'] == 'No')].shape[0]

female_suicidal_counts = data[(data['Gender'] == 'Female') & (data['Have you ever had suicidal thoughts ?'] == 'Yes')].shape[0]
female_no_suicidal_counts = data[(data['Gender'] == 'Female') & (data['Have you ever had suicidal thoughts ?'] == 'No')].shape[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))  # 1 row, 2 columns

labels = ['Suicidal Thoughts', 'No Suicidal Thoughts']
sizes_male = [male_suicidal_counts, male_no_suicidal_counts]
colors = ['lightcoral', 'lightskyblue']
explode = (0.1, 0)

ax1.pie(sizes_male, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
ax1.set_title('Proportion of Males with and without Suicidal Thoughts')
ax1.axis('equal')

sizes_female = [female_suicidal_counts, female_no_suicidal_counts]
ax2.pie(sizes_female, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
ax2.set_title('Proportion of Females with and without Suicidal Thoughts')
ax2.axis('equal')

plt.tight_layout()
plt.show()


In [ ]:
# Study Hours & Depression
plt.figure(figsize=(10, 6))
sns.boxplot(x='Depression', y='Study Hours', data=data , palette=colors)
plt.title('Study Hours vs Depression')
plt.show()

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
study_hours_depression = data.groupby(['Study Hours', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
study_hours_percent = study_hours_depression.div(study_hours_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
study_hours_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Study Hours vs Depression (Stacked Bar Chart)')
plt.xlabel('Study Hours')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = study_hours_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
data.columns

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
study_hours_depression = data.groupby(['Sleep Hours Category', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
study_hours_percent = study_hours_depression.div(study_hours_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
study_hours_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Study Hours vs Depression (Stacked Bar Chart)')
plt.xlabel('Study Hours')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = study_hours_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
study_hours_depression = data.groupby(['Sleep Hours Category', 'Depression']).size().unstack(fill_value=0)

# เรียงลำดับข้อมูลตามที่ต้องการ
desired_order = ['0-3', '4-6', '7-9', '10-12']
study_hours_depression = study_hours_depression.reindex(desired_order, axis=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
study_hours_percent = study_hours_depression.div(study_hours_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
study_hours_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Study Hours vs Depression (Stacked Bar Chart)')
plt.xlabel('Study Hours')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = study_hours_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
data['Study Hours'].dtype

In [ ]:
data.columns

In [ ]:
#เช็คจำนวนข้อมูลที่เหมือนกันของ Have you ever had suicidal thoughts ? กับ Family History of Mental Illness
data['Comparison'] = data['Have you ever had suicidal thoughts ?'] == data['Family History of Mental Illness']

#นับจำนวนข้อมูล
true_count = data['Comparison'].value_counts()[True]
false_count = data['Comparison'].value_counts()[False]

print(f"Number of True values: {true_count}")
print(f"Number of False values: {false_count}")

In [ ]:
data

In [ ]:
# Family History of Mental Illness & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Family History of Mental Illness', hue='Depression', data=data)
plt.title('Family History of Mental Illness vs Depression')
plt.show()

In [ ]:
# Financial Stress & Depression
plt.figure(figsize=(10, 6))
sns.countplot(x='Financial Stress', hue='Depression', data=data)
plt.title('Financial Stress vs Depression')
plt.show()

In [ ]:
# คำนวณจำนวนคนแต่ละกลุ่ม
Financial_Stress_depression = data.groupby(['Financial Stress', 'Depression']).size().unstack(fill_value=0)

# คำนวณเปอร์เซ็นต์ของแต่ละกลุ่ม
Financial_Stress_percent = Financial_Stress_depression.div(Financial_Stress_depression.sum(axis=1), axis=0) * 100

# สร้าง Stacked Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
Financial_Stress_depression.plot(kind='bar', stacked=True, ax=ax, color=['lightskyblue', 'lightcoral'])

plt.title('Financial Stress vs Depression (Stacked Bar Chart)')
plt.xlabel('Financial Stress')
plt.ylabel('Number of Individuals')  # แกน Y เป็นจำนวนคน
plt.xticks(rotation=0)
plt.legend(title='Depression')

# แสดงเปอร์เซ็นต์ภายในแท่งเดียวกัน
for i, bars in enumerate(ax.containers):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + height / 2
            percent = Financial_Stress_percent.iloc[int(x), i]  # ดึงค่าจาก DataFrame
            ax.annotate(f'{percent:.1f}%', (x, y), ha='center', va='center', color='white', fontsize=10)

plt.tight_layout()
plt.show()

# Logistic Regression

In [ ]:
# split X and y into training and testing sets
from sklearn.model_selection import train_test_split
X = data.drop('Depression', axis=1)
y = data['Depression']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
# import the class
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# instantiate the model (using the default parameters)
model = LogisticRegression(random_state=16)

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Apply label encoding to categorical columns in X_train and X_test
for col in X_train.select_dtypes(include=['object']).columns:
    X_train[col] = label_encoder.fit_transform(X_train[col])
    X_test[col] = label_encoder.transform(X_test[col])

# fit the model with data
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
# import the metrics class
from sklearn import metrics

cnf_matrix = metrics.confusion_matrix(y_test, y_pred)
cnf_matrix